## Self Intruction Fine Tunning Method

In [1]:
import torch
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments

# === Step 1: Load CoLA Dataset ===
print("Loading CoLA dataset...")
dataset = load_dataset("glue", "cola")
train_dataset = dataset["train"].shuffle(seed=42).select(range(500))  # Use a subset for demonstration
validation_dataset = dataset["validation"].shuffle(seed=42).select(range(100))

Loading CoLA dataset...


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1043 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1063 [00:00<?, ? examples/s]

In [2]:
# === Step 2: Tokenize Inputs ===
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=64)

print("Tokenizing dataset...")
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_validation_dataset = validation_dataset.map(tokenize_function, batched=True)

# Remove unnecessary columns and set format for PyTorch
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["sentence", "idx"]).with_format("torch")
tokenized_validation_dataset = tokenized_validation_dataset.remove_columns(["sentence", "idx"]).with_format("torch")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing dataset...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [3]:
# === Step 3: Generate Synthetic Data ===
print("Generating synthetic instruction-response pairs...")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

def generate_synthetic_data(dataset):
    synthetic_data = []
    for example in dataset:
        inputs = {key: example[key].unsqueeze(0) for key in ["input_ids", "attention_mask"]}
        with torch.no_grad():
            outputs = model(**inputs)
            predicted_label = torch.argmax(outputs.logits, dim=-1).item()
        
        # Create synthetic instruction-response pair
        instruction = f"Classify the grammaticality of the following sentence: '{tokenizer.decode(example['input_ids'], skip_special_tokens=True)}'"
        response = "Grammatical" if predicted_label == 1 else "Ungrammatical"
        synthetic_data.append({"instruction": instruction, "response": response})
    
    return synthetic_data

synthetic_train_data = generate_synthetic_data(tokenized_train_dataset)

# Convert synthetic data back to tokenized format
def tokenize_synthetic_data(data):
    tokenized_data = []
    for item in data:
        encoded = tokenizer(item["instruction"], item["response"], padding="max_length", truncation=True, max_length=128)
        tokenized_data.append(encoded)
    return tokenized_data

tokenized_synthetic_train_data = tokenize_synthetic_data(synthetic_train_data)

Generating synthetic instruction-response pairs...


model.safetensors:  31%|###1      | 83.9M/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# === Step 4: Prepare Model for Fine-Tuning ===
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# === Step 5: Define Training Arguments ===
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# === Step 6: Fine-Tune the Model ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_synthetic_train_data,
    eval_dataset=tokenized_validation_dataset,
    tokenizer=tokenizer,
)

print("Starting fine-tuning...")
trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/aliakbar/miniconda3/envs/nlpllm/lib/python3.11/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>={ACCELERATE_MIN_VERSION}'`

In [ ]:
# === Step 7: Evaluate the Model ===
print("Evaluating model...")
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# === Step 8: Save the Fine-Tuned Model ===
print("Saving fine-tuned model...")
model.save_pretrained("./fine_tuned_self_instruct_distilbert")
tokenizer.save_pretrained("./fine_tuned_self_instruct_distilbert")